# Averitec Submission Workflow

This notebook runs `facticli` on an Averitec-formatted input file and writes Averitec submission outputs (`.json` and `.jsonl`).

## 1) Configure Paths and Runtime

Edit these values before running the notebook.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repository root (missing pyproject.toml in parent paths).")


REPO_ROOT = find_repo_root()
print(f"Repo root: {REPO_ROOT}")

INPUT_PATH = REPO_ROOT / "data/averitec/dev_20.json"
OUTPUT_JSON = REPO_ROOT / "data/averitec/submission_dev20.json"
OUTPUT_JSONL = REPO_ROOT / "data/averitec/submission_dev20.jsonl"

INFERENCE_PROVIDER = "openai"  # openai | gemini | openai-agents
SEARCH_PROVIDER = "openai"     # openai | brave
MODEL = None                       # e.g., "gpt-4.1-mini"

MAX_CHECKS = 4
PARALLEL_RESEARCH = 4
PARALLEL_CLAIMS = 1
MAX_EVIDENCE = 10

OFFSET = 0
LIMIT = None
EMPTY_QUESTION = False
FAIL_FAST = False

print(f"Input:  {INPUT_PATH}")
print(f"Output: {OUTPUT_JSON}")
print(f"Output: {OUTPUT_JSONL}")


## 2) Load `.env` (Optional) and Validate Required Keys

In [ ]:
env_path = REPO_ROOT / '.env'
if env_path.exists():
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())
    print(f"Loaded environment defaults from {env_path}")
else:
    print('.env not found; using current shell environment.')

required_key = 'OPENAI_API_KEY' if INFERENCE_PROVIDER in {'openai', 'openai-agents'} else 'GEMINI_API_KEY'
if not os.getenv(required_key):
    raise RuntimeError(f"Missing required env var: {required_key}")
if SEARCH_PROVIDER == 'brave' and not os.getenv('BRAVE_SEARCH_API_KEY'):
    raise RuntimeError('Missing required env var: BRAVE_SEARCH_API_KEY')

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")

print('Environment validation passed.')


## 3) Run `facticli` Averitec Submission Pipeline

In [ ]:
python_bin = REPO_ROOT / '.venv/bin/python'
if not python_bin.exists():
    python_bin = Path(sys.executable)

cmd = [
    str(python_bin),
    str(REPO_ROOT / 'scripts/run_averitec_submission.py'),
    '--input', str(INPUT_PATH),
    '--output', str(OUTPUT_JSON),
    '--inference-provider', INFERENCE_PROVIDER,
    '--search-provider', SEARCH_PROVIDER,
    '--max-checks', str(MAX_CHECKS),
    '--parallel', str(PARALLEL_RESEARCH),
    '--parallel-claims', str(PARALLEL_CLAIMS),
    '--max-evidence', str(MAX_EVIDENCE),
    '--offset', str(OFFSET),
]

if LIMIT is not None:
    cmd.extend(['--limit', str(LIMIT)])
if MODEL:
    cmd.extend(['--model', MODEL])
if EMPTY_QUESTION:
    cmd.append('--empty-question')
if FAIL_FAST:
    cmd.append('--fail-fast')

print('Running command:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)
print(f'Finished. Wrote JSON submission to {OUTPUT_JSON}')


## 4) Convert JSON Array to JSONL and Summarize

In [ ]:
rows = json.loads(OUTPUT_JSON.read_text(encoding='utf-8'))
if not isinstance(rows, list):
    raise TypeError('Expected JSON array submission.')

OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_JSONL.open('w', encoding='utf-8') as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

label_counts = Counter(row.get('pred_label', '<missing>') for row in rows)
print(f'Rows written: {len(rows)}')
print(f'JSONL path:   {OUTPUT_JSONL}')
print('Label counts:')
for label, count in label_counts.items():
    print(f'  {label}: {count}')

print('\nFirst output row:')
print(json.dumps(rows[0], indent=2, ensure_ascii=False)[:2000])
